In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
dynamic_file = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_04_withBias_1h_dyn_low_high.txt"

df = pd.read_csv(dynamic_file)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (360000, 18)
['IMU_low_ax', 'IMU_low_ay', 'IMU_low_az', 'IMU_low_gx', 'IMU_low_gy', 'IMU_low_gz', 'IMU_low_mx', 'IMU_low_my', 'IMU_low_mz', 'IMU_high_ax', 'IMU_high_ay', 'IMU_high_az', 'IMU_high_gx', 'IMU_high_gy', 'IMU_high_gz', 'IMU_high_mx', 'IMU_high_my', 'IMU_high_mz']


In [4]:
for axis in ["x", "y", "z"]:
    df[f"error_{axis}"] = (
        df[f"IMU_low_a{axis}"] -
        df[f"IMU_high_a{axis}"]
    )

target_cols = ["error_x", "error_y", "error_z"]

print(df[target_cols].describe())

             error_x        error_y        error_z
count  360000.000000  360000.000000  360000.000000
mean       -2.005781       2.106845       0.965404
std         0.167525       0.179763       0.168619
min        -2.438300       1.661800       0.466920
25%        -2.141368       1.965368       0.831000
50%        -2.008600       2.106100       0.964700
75%        -1.868845       2.244700       1.102000
max        -1.584390       2.649480       1.374440


In [5]:
accel_features = [
    "IMU_low_ax",
    "IMU_low_ay",
    "IMU_low_az"
]

X_accel = df[accel_features].values
Y = df[target_cols].values

split = int(0.8 * len(df))

X_train = X_accel[:split]
X_test = X_accel[split:]

Y_train = Y[:split]
Y_test = Y[split:]

scaler_accel = StandardScaler()

X_train_s = scaler_accel.fit_transform(X_train)
X_test_s = scaler_accel.transform(X_test)

model_accel = Ridge(alpha=1.0)

model_accel.fit(X_train_s, Y_train)

pred_accel = model_accel.predict(X_test_s)

In [6]:
multisensor_features = [
    "IMU_low_ax",
    "IMU_low_ay",
    "IMU_low_az",
    "IMU_low_gx",
    "IMU_low_gy",
    "IMU_low_gz"
]

X_multi = df[multisensor_features].values

X_train_multi = X_multi[:split]
X_test_multi = X_multi[split:]

scaler_multi = StandardScaler()

X_train_multi_s = scaler_multi.fit_transform(X_train_multi)
X_test_multi_s = scaler_multi.transform(X_test_multi)

model_multi = Ridge(alpha=1.0)

model_multi.fit(
    X_train_multi_s,
    Y_train
)

pred_multi = model_multi.predict(
    X_test_multi_s
)

In [7]:
for i, axis in enumerate(["X", "Y", "Z"]):

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            np.full(len(Y_test), Y_train[:, i].mean())
        )
    )

    accel_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            pred_accel[:, i]
        )
    )

    multi_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            pred_multi[:, i]
        )
    )

    accel_mae = mean_absolute_error(
        Y_test[:, i],
        pred_accel[:, i]
    )

    multi_mae = mean_absolute_error(
        Y_test[:, i],
        pred_multi[:, i]
    )

    accel_improvement = (
        1 - accel_rmse / baseline_rmse
    ) * 100

    multi_improvement = (
        1 - multi_rmse / baseline_rmse
    ) * 100

    print(f"\n{axis}-axis")
    print(f"Baseline RMSE       : {baseline_rmse:.6f}")
    print(f"Accel-only RMSE     : {accel_rmse:.6f}")
    print(f"Accel-only MAE      : {accel_mae:.6f}")
    print(f"Accel improvement   : {accel_improvement:.2f}%")
    print(f"Accel+Gyro RMSE     : {multi_rmse:.6f}")
    print(f"Accel+Gyro MAE      : {multi_mae:.6f}")
    print(f"Multi improvement   : {multi_improvement:.2f}%")


X-axis
Baseline RMSE       : 0.187237
Accel-only RMSE     : 0.098545
Accel-only MAE      : 0.089417
Accel improvement   : 47.37%
Accel+Gyro RMSE     : 0.098551
Accel+Gyro MAE      : 0.089418
Multi improvement   : 47.37%

Y-axis
Baseline RMSE       : 0.181475
Accel-only RMSE     : 0.084489
Accel-only MAE      : 0.066965
Accel improvement   : 53.44%
Accel+Gyro RMSE     : 0.084555
Accel+Gyro MAE      : 0.067010
Multi improvement   : 53.41%

Z-axis
Baseline RMSE       : 0.191492
Accel-only RMSE     : 0.110773
Accel-only MAE      : 0.094106
Accel improvement   : 42.15%
Accel+Gyro RMSE     : 0.110818
Accel+Gyro MAE      : 0.094132
Multi improvement   : 42.13%
